In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, StandardScaler, RobustScaler, KBinsDiscretizer



df = pd.read_excel("data/Online_Retail.xlsx")

print(df.shape)

# checking too see what has missing values  
df.isna().sum()

(541909, 8)


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [ ]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, StandardScaler, RobustScaler, KBinsDiscretizer



df = pd.read_excel("data/Online_Retail.xlsx")

# Text Preprocessing 
df['Description'] = df['Description'].str.lower()
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.lower()
df['CustomerID'] = df['CustomerID'].astype(str)

# Split between Test and Train
X_train, X_test, = train_test_split(
    df, test_size=0.2, random_state=21, 
)

# Pipeline & Column Transformers 
categorical_features = ["InvoiceNo", "StockCode", "Description", "InvoiceDate","CustomerID","Country"]

# X_train[categorical_features] = X_train[categorical_features].astype(str).str.lower()
# X_test[categorical_features] = X_test[categorical_features].astype(str).str.lower()

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='constant',fill_value= 'Unknown')),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, categorical_features),
    ]
)

preprocessor.fit(X_train).transform(X_test)

The matrix is still rather large, I'm going to reduce the size by deriving from the nominal values too help compress the data while stil maintaining meaning. 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, StandardScaler, RobustScaler, KBinsDiscretizer



df = pd.read_excel("data/Online_Retail.xlsx")

# cleaning Description 
df['Description'] = (
    df['Description']
      .str.lower()
      .str.replace(r'[^\w\s]', ' ', regex=True)
      .str.replace(r'[\s+]', ' ', regex=True)
      .str.strip()
      .replace('', np.nan)
)

# Derive isCancellation from invoice 
df['IsCancellation'] = df['InvoiceNo'].astype(str).str.upper().str.startswith('C')

# Derive Month and DayOfWeek 

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Month'] = df['InvoiceDate'].dt.month          
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek  

# Derive HasCustomerID
df['HasCustomerID'] = df['CustomerID'].notna()

# converet derived strings too int
df['IsCancellation'] = df['IsCancellation'].astype(int)
df['HasCustomerID'] = df['HasCustomerID'].astype(int)


# X_train[categorical_features] = X_train[categorical_features].astype(str)
# X_test[categorical_features] = X_test[categorical_features].astype(str)

# Split between Test and Train
X_train, X_test, = train_test_split(
    df, test_size=0.2, random_state=21, 
)

# Pipeline & Column Transformers 
categorical_features = ['Description', 'Country', 'Month', 'DayOfWeek', 'IsCancellation', 'HasCustomerID']

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='constant',fill_value= 'Unknown')),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipe, categorical_features),
    ],remainder='drop'
)

preprocessor.fit(X_train).transform(X_test)

Challenge C:
The primary issue that occurs with a dimple matching coeffcient is the fact that mutaul abscence is used in its calculation. This will severly overestimate the similarty between two objects. This can be avoided by using Jaccard Coeffcient, Jaccard does not use Mutual absence in its calcualtion. This means that the similarity is based purely on items that would exist in both carts. This differs from SMC which would factor in not only items that are in both carts but also all the items that the carts did not share. 

Challenge A: 
